In [ ]:
import glob, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Load all daily draw CSVs ──────────────────────────────────────────────────
files = glob.glob('daily_draws/*.csv')
daily_raw = pd.concat([
    pd.read_csv(f).assign(draw_date=os.path.splitext(os.path.basename(f))[0])
    for f in files
])
daily_raw.rename(columns={'time [UTC]': 'date', 'value': 'daily'}, inplace=True)
daily_raw['date']      = pd.to_datetime(daily_raw['date'],      format='mixed').dt.date
daily_raw['draw_date'] = pd.to_datetime(
    daily_raw['draw_date'].str.replace('draw_', ''), format='mixed'
).dt.date
daily_pivot    = daily_raw.pivot_table(index='date', columns='draw_date',
                                       values='daily', aggfunc='first')
daily_draw_cols = daily_pivot.columns.tolist()

# ── Weekly benchmarks (rescaled final series) ────────────────────────────────
weekly_final = pivot[draw_cols].mean(axis=1)
weekly_final = weekly_final / weekly_final.max() * 100

# ── Scale reconciliation ─────────────────────────────────────────────────────
# Assumption: weekly_gtrends[w] = mean(daily[w]) * C  (C unknown constant).
# Estimate C by aligning on the full-sample mean across all benchmark weeks.
def estimate_scale_constant(indicator: pd.Series,
                             benchmarks: pd.Series) -> float:
    """
    Estimate the multiplicative constant C such that
        benchmark[w] ≈ C * within-week mean of indicator.
    Uses the ratio of means across all benchmark weeks (full-sample alignment).
    """
    weekly_means = []
    for wk in benchmarks.index:
        wk_end   = pd.Timestamp(wk)
        wk_start = wk_end - pd.Timedelta(days=6)
        mask     = (pd.to_datetime(indicator.index) >= wk_start) & \
                   (pd.to_datetime(indicator.index) <= wk_end)
        vals = indicator[mask]
        if len(vals) > 0 and not np.isnan(vals.mean()):
            weekly_means.append(vals.mean())
        else:
            weekly_means.append(np.nan)
    weekly_means = pd.Series(weekly_means, index=benchmarks.index)
    valid = weekly_means.notna()
    return float(benchmarks[valid].mean() / weekly_means[valid].mean())


# ── Denton proportional adjustment (FIXED) ───────────────────────────────────
# FIX: denominator is within-week MEAN, not sum.
# Benchmark = mean(daily) * C  →  pro-rata factor = benchmark / mean(daily).
# Using sum would implicitly assume benchmark = sum(daily), which is wrong for
# the Google Trends normalisation and discards cross-week structure.
def denton_proportional(indicator: pd.Series,
                         benchmarks: pd.Series) -> pd.Series:
    """
    Proportional Denton adjustment.
    For each benchmark week w:
        adjusted[w] = daily[w] * (benchmarks[w] / mean(daily[w]))
    benchmarks must be a Series indexed by week-end dates.
    """
    result = indicator.copy().astype(float)
    for wk in benchmarks.index:
        wk_end   = pd.Timestamp(wk)
        wk_start = wk_end - pd.Timedelta(days=6)
        mask     = (pd.to_datetime(result.index) >= wk_start) & \
                   (pd.to_datetime(result.index) <= wk_end)
        daily_vals = result[mask]
        if len(daily_vals) == 0 or daily_vals.mean() == 0:
            continue
        # Use .loc[wk] for safe label-based lookup (avoids positional misalignment)
        result[mask] = daily_vals * (benchmarks.loc[wk] / daily_vals.mean())
    return result


# ── Method 1: Naive average + Denton ─────────────────────────────────────────
naive_avg = daily_pivot[daily_draw_cols].mean(axis=1)

# Scale reconciliation before Denton
C_naive   = estimate_scale_constant(naive_avg, weekly_final)
print(f"Scale constant (naive avg):    C = {C_naive:.4f}")
naive_avg_rescaled = naive_avg * C_naive

naive_denton = denton_proportional(naive_avg_rescaled, weekly_final)
naive_denton = naive_denton / naive_denton.max() * 100


# ── Method 2: Rescale to consensus peak, then average + Denton ───────────────
daily_peaks          = daily_pivot[daily_draw_cols].idxmax()
consensus_peak_daily = daily_peaks.mode()[0]
n_agree = (daily_peaks == consensus_peak_daily).sum()
print(f"Daily consensus peak: {consensus_peak_daily}  ({n_agree}/{len(daily_draw_cols)} draws)")

daily_rescaled = daily_pivot[daily_draw_cols].copy().astype(float)
for col in daily_draw_cols:
    if daily_peaks[col] != consensus_peak_daily:
        val = daily_rescaled.loc[consensus_peak_daily, col]
        if val > 0:
            daily_rescaled[col] = daily_rescaled[col] * (100.0 / val)
            print(f"  Rescaled {col} (peaked at {daily_peaks[col]}, "
                  f"had {val:.1f} at consensus)")

rescaled_avg = daily_rescaled.mean(axis=1)

# Scale reconciliation before Denton
C_rescaled   = estimate_scale_constant(rescaled_avg, weekly_final)
print(f"Scale constant (rescaled avg): C = {C_rescaled:.4f}")
rescaled_avg_sc = rescaled_avg * C_rescaled

rescaled_denton = denton_proportional(rescaled_avg_sc, weekly_final)
rescaled_denton = rescaled_denton / rescaled_denton.max() * 100


# ── Plot comparison ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(naive_denton.index,    naive_denton.values,
        color='red',      lw=1.5, ls='--', label='Naive avg + Denton')
ax.plot(rescaled_denton.index, rescaled_denton.values,
        color='darkblue', lw=2,           label='Rescaled avg + Denton')
ax.set(xlabel='Date', ylabel='Index (max = 100)',
       title=f'Daily Draws: Both Denton-adjusted ({len(daily_draw_cols)} draws)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# ── Quantify the difference ───────────────────────────────────────────────────
diff = (rescaled_denton - naive_denton).abs()
print(f"\nMax absolute difference:  {diff.max():.2f}")
print(f"Mean absolute difference: {diff.mean():.2f}")

# Data Science Tools and Ecosystem


In this notebook, Data Science Tools and Ecosystem are summarized.

**Objectives:**

- List popular languages for Data Science.
- Introduce commonly used libraries in Data Science.
- Explore examples of evaluating arithmetic expressions in Python.
- Understand how to convert minutes to hours using Python.
- Summarize the Data Science tools and ecosystem.


Some of the popular languages that Data Scientists use are:

1. Python
2. R
3. SQL
4. Julia
5. Scala


Some of the commonly used libraries by Data Scientists include:

1. NumPy
2. Pandas
3. Matplotlib
4. Scikit-Learn
5. TensorFlow


| Data Science Tools    |
|-----------------------|
| Jupyter Notebook      |
| RStudio               |
| Visual Studio Code    |


### Examples of Evaluating Arithmetic Expressions in Python

In Python, you can perform various arithmetic operations. Here are some examples:

1. **Addition:**
   ```python
   result = 5 + 3
   # result will be 8
result = 10 - 4
# result will be 6


In [6]:
# This a simple arithmetic expression to mutiply then add integers
(3*4)+5

17

In [8]:
#This will convert 200 minutes to hours by diving by 60
200/60

3.3333333333333335

## Author
Andrea Lamacchia


## Robustness Check: Denton on Naive Average vs Denton on Rescaled Average

**Working assumption:** weekly Gtrends = within-week **mean** of the underlying daily query share × an unknown multiplicative constant.  
The constant is estimated by aligning the two series on the full-sample mean (scale reconciliation) before applying the Denton proportional adjustment.  
Pro-rata via weekly **sum** (as often coded by default) is wrong here and throws away cross-week information.

Two methods compared:
1. **Naive avg + Denton** — average the raw daily draws, reconcile scale, apply Denton.
2. **Rescaled avg + Denton** — rescale each draw to a consensus peak first, then average, reconcile scale, apply Denton.